# 00 · 纯云端高速同步开源视频模型到 5TB Google Drive

**全云端零本地占用**：本 Notebook 走 Google 机房千兆内网直接将 Hugging Face 权重拉进你的 **5TB Google Drive (`/MyDrive/AI_Video/models/`)**，不占本地硬盘、不走本地宽带。

**省算力提示**：请在右上角选择 **免费 CPU 运行时** 执行（消耗 `0 CU`）。

**默认预选（MiniMax H3 全能主力 + Wan 2.2 商用快出，合计约 115 GB，仅占 5TB 空间 2.3%）**：
- `minimax_h3_fl2va_a100` + `minimax_h3_ref2va_a100` + `minimax_h3_turbo_lora`（原生 32kHz 立体声 / 最长 15s / 多图锁角色）
- `wan22_i2v_fp8` + `wan22_lightx2v_lora`（Apache 2.0 商用无忧 / 4步极速批量出片）

In [ ]:
#@title 1. 挂载 5TB Google Drive 并拉取最新仓库代码
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

!pip install -q "huggingface_hub[hf_transfer]"
!python {REPO_DIR}/scripts/sync_models_to_drive.py --manifest {REPO_DIR}/configs/models_manifest.json --packs dummy --list


In [ ]:
#@title 2. 勾选需要云对云同步到 5TB Google Drive 的模型包
MINIMAX_H3_FL2VA_A100 = True #@param {type:"boolean"}
MINIMAX_H3_REF2VA_A100 = True #@param {type:"boolean"}
MINIMAX_H3_TURBO_LORA = True #@param {type:"boolean"}
WAN22_I2V_FP8 = False #@param {type:"boolean"}
WAN22_LIGHTX2V_LORA = False #@param {type:"boolean"}
FLUX2_KEYFRAME = False #@param {type:"boolean"}
WAN22_T2V_FP8 = False #@param {type:"boolean"}
LTX25_AV_A100 = False #@param {type:"boolean"}
FORCE_REDOWNLOAD = False #@param {type:"boolean"}
HF_TOKEN_INPUT = "" #@param {type:"string"}

import os, subprocess

hf_token = HF_TOKEN_INPUT.strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        hf_token = ''

packs = []
if MINIMAX_H3_FL2VA_A100: packs.append('minimax_h3_fl2va_a100')
if MINIMAX_H3_REF2VA_A100: packs.append('minimax_h3_ref2va_a100')
if MINIMAX_H3_TURBO_LORA: packs.append('minimax_h3_turbo_lora')
if WAN22_I2V_FP8: packs.append('wan22_i2v_fp8')
if WAN22_LIGHTX2V_LORA: packs.append('wan22_lightx2v_lora')
if FLUX2_KEYFRAME: packs.append('flux2_keyframe')
if WAN22_T2V_FP8: packs.append('wan22_t2v_fp8')
if LTX25_AV_A100: packs.append('ltx25_av_a100')

if not packs:
    raise ValueError('请至少勾选一个模型包！')

cmd = [
    'python', f'{REPO_DIR}/scripts/sync_models_to_drive.py',
    '--manifest', f'{REPO_DIR}/configs/models_manifest.json',
    '--packs', *packs,
]
if FORCE_REDOWNLOAD:
    cmd.append('--force')
if hf_token:
    cmd.extend(['--hf-token', hf_token])

print('执行命令:', ' '.join(cmd[:-2] + ['--hf-token', '***'] if hf_token else cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 3. 检查 5TB Google Drive 中已就绪的模型清单与磁盘占用
!du -sh /content/drive/MyDrive/AI_Video/models/*
!find /content/drive/MyDrive/AI_Video/models -type f -exec ls -lh {} +
